In [2]:
m=Municipio.objects.get(municipio=3)
m.seccion_set.all()

<QuerySet [<Seccion: 01 003 0012>, <Seccion: 01 003 0013>, <Seccion: 01 003 0014>, <Seccion: 01 003 0015>, <Seccion: 01 003 0016>, <Seccion: 01 003 0017>, <Seccion: 01 003 0018>, <Seccion: 01 003 0019>, <Seccion: 01 003 0020>, <Seccion: 01 003 0021>, <Seccion: 01 003 0022>, <Seccion: 01 003 0023>, <Seccion: 01 003 0024>, <Seccion: 01 003 0025>, <Seccion: 01 003 0026>, <Seccion: 01 003 0027>, <Seccion: 01 003 0028>, <Seccion: 01 003 0029>, <Seccion: 01 003 0030>, <Seccion: 01 003 0031>, '...(remaining elements truncated)...']>

In [6]:
import os
from os import scandir, getcwd
import datetime
import shutil
import PyPDF2
from control.models import Distrito, Municipio, Seccion, Pusinex2
from django.contrib.auth.models import User


def ls(ruta = getcwd()):
    return [arch.name for arch in scandir(ruta) if arch.is_file()]

user = User.objects.get(id=1)
now = datetime.datetime.now()

DISTRITO_OLD = 3

baseOld = os.path.join(getcwd(), '..', 'media', 'pusinex-old')
baseNew = os.path.join(getcwd(), '..', 'media', 'pusinex-new')
ext = 'pdf'


for p in ls(f'../media/pusinex-old/0{DISTRITO_OLD}'):
    try:
        seccion = Seccion.objects.get(seccion=int(p[4:8]))
        municipio = seccion.municipio
        distrito = seccion.distrito
        entidad = distrito.entidad

        # Verificamos sin el pusinex tiene localidad para establecer la fecha de revisión
        idx = 17 if p[8] == '-' else 12
        rev = datetime.datetime(int(p[idx:idx+4]), int(p[idx+4:idx+6]), int(p[idx+6:idx+8]))
        nombre = f'{entidad.entidad:02}{distrito.distrito:02}{municipio.municipio:02}{seccion.seccion:04}_rev{rev:%Y%m%d}.{ext}'
        rutaNew = os.path.join(baseNew, f'{distrito.distrito:02}', nombre)
        rutaOld = os.path.join(baseOld, f'{DISTRITO_OLD:02}', p)
        hojas = len(PyPDF2.PdfReader(rutaOld).pages)
        archivo = f'pusinex/{distrito.distrito:02}/{nombre}'

        print(archivo, f'Hojas: {hojas}')
        shutil.copy(rutaOld, rutaNew)
    
        pusinex = Pusinex2()
        pusinex.seccion = seccion
        pusinex.f_act = rev
        pusinex.hojas = hojas
        pusinex.archivo = archivo
        pusinex.user = user
        pusinex.created = now
        pusinex.updated = now
        pusinex.save()

    except:
        print(f'No existe la sección {p[4:8]}')

pusinex/02/2902010002_rev20221214.pdf Hojas: 4
pusinex/02/2902060078_rev20221214.pdf Hojas: 7
pusinex/02/2902060085_rev20221214.pdf Hojas: 2
pusinex/02/2902060087_rev20230328.pdf Hojas: 1
pusinex/02/2902060089_rev20230328.pdf Hojas: 2
pusinex/02/2902150234_rev20221214.pdf Hojas: 2
pusinex/02/2902150242_rev20221214.pdf Hojas: 3
pusinex/02/2902190290_rev20221214.pdf Hojas: 2
pusinex/02/2902200297_rev20230328.pdf Hojas: 3
pusinex/03/2903460314_rev20221214.pdf Hojas: 6
pusinex/02/2902230325_rev20230328.pdf Hojas: 2
pusinex/02/2902240336_rev20230328.pdf Hojas: 6
pusinex/03/2903320425_rev20230328.pdf Hojas: 2
pusinex/02/2902360511_rev20230328.pdf Hojas: 5
pusinex/02/2902360512_rev20230328.pdf Hojas: 9
pusinex/02/2902360515_rev20230328.pdf Hojas: 5
pusinex/02/2902570551_rev20230328.pdf Hojas: 5
pusinex/02/2902430575_rev20230328.pdf Hojas: 3
pusinex/02/2902430581_rev20230328.pdf Hojas: 7
pusinex/03/2903440589_rev20230328.pdf Hojas: 2
pusinex/03/2903440593_rev20230328.pdf Hojas: 3
pusinex/03/29

In [5]:
Seccion.objects.all().order_by('distrito', 'municipio', ).filter(distrito=2)

<QuerySet [<Seccion: 02 001 0001>, <Seccion: 02 001 0002>, <Seccion: 02 001 0003>, <Seccion: 02 001 0004>, <Seccion: 02 002 0005>, <Seccion: 02 002 0006>, <Seccion: 02 002 0007>, <Seccion: 02 002 0008>, <Seccion: 02 002 0009>, <Seccion: 02 002 0010>, <Seccion: 02 002 0011>, <Seccion: 02 003 0038>, <Seccion: 02 006 0078>, <Seccion: 02 006 0079>, <Seccion: 02 006 0080>, <Seccion: 02 006 0081>, <Seccion: 02 006 0082>, <Seccion: 02 006 0083>, <Seccion: 02 006 0084>, <Seccion: 02 006 0085>, '...(remaining elements truncated)...']>

In [13]:
mpios=Municipio.objects.all()
m=mpios[0]
m.seccion_set.all()[0].distrito.distrito

2

In [30]:
d =  Distrito.objects.all().prefetch_related()
d1 = d[0]
d.values()

<QuerySet [{'entidad_id': 29, 'distrito': 1, 'cabecera': 'APIZACO'}, {'entidad_id': 29, 'distrito': 2, 'cabecera': 'TLAXCALA DE XICOHTENCATL'}, {'entidad_id': 29, 'distrito': 3, 'cabecera': 'ZACATELCO'}]>

In [29]:
for s in d1.seccion_set.filter(activa=True).order_by('municipio', 'seccion'):
    print(s.distrito, s.municipio, s.seccion)

29-01 003 APIZACO 12
29-01 003 APIZACO 13
29-01 003 APIZACO 14
29-01 003 APIZACO 15
29-01 003 APIZACO 16
29-01 003 APIZACO 17
29-01 003 APIZACO 18
29-01 003 APIZACO 19
29-01 003 APIZACO 20
29-01 003 APIZACO 21
29-01 003 APIZACO 22
29-01 003 APIZACO 23
29-01 003 APIZACO 24
29-01 003 APIZACO 25
29-01 003 APIZACO 26
29-01 003 APIZACO 27
29-01 003 APIZACO 28
29-01 003 APIZACO 29
29-01 003 APIZACO 31
29-01 003 APIZACO 32
29-01 003 APIZACO 33
29-01 003 APIZACO 34
29-01 003 APIZACO 35
29-01 003 APIZACO 36
29-01 003 APIZACO 37
29-01 003 APIZACO 39
29-01 003 APIZACO 40
29-01 003 APIZACO 42
29-01 003 APIZACO 43
29-01 003 APIZACO 44
29-01 003 APIZACO 45
29-01 003 APIZACO 46
29-01 003 APIZACO 47
29-01 003 APIZACO 48
29-01 003 APIZACO 618
29-01 003 APIZACO 619
29-01 003 APIZACO 634
29-01 003 APIZACO 635
29-01 003 APIZACO 636
29-01 004 ATLANGATEPEC 49
29-01 004 ATLANGATEPEC 50
29-01 004 ATLANGATEPEC 51
29-01 004 ATLANGATEPEC 52
29-01 004 ATLANGATEPEC 53
29-01 004 ATLANGATEPEC 54
29-01 004 ATLANGATEP

In [40]:
from django.db.models import Count, Q, F


secciones = Seccion.objects.filter(activa=True).order_by('distrito', 'municipio', 'seccion')
distritos = Distrito.objects.all()
distritos

<QuerySet [<Distrito: 29-01>, <Distrito: 29-02>, <Distrito: 29-03>]>